In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv


In [2]:
!pip install -q mlflow dagshub

import dagshub
import mlflow

DAGSHUB_USER = 'gormo22'
REPO_NAME = 'ML-HousePricePrediction'

dagshub.init(repo_owner=DAGSHUB_USER, repo_name=REPO_NAME, mlflow=True)
tracking_uri = f"https://dagshub.com/{DAGSHUB_USER}/{REPO_NAME}.mlflow"
mlflow.set_tracking_uri(tracking_uri)

mlflow.set_experiment("House_Price_Prediction")
print("MLflow and DagsHub successfully connected.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 67.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.5

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=35f2af61-13cb-4fcc-abfd-4b6d9bf5c3a8&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=6514d008ef5b2cec29eb771f2612e8889229d9a27d20d14c1dd54ea1387b5879




Accessing as gormo22

Initialized MLflow to track repo "gormo22/ML-HousePricePrediction"

Repository gormo22/ML-HousePricePrediction initialized!

MLflow and DagsHub successfully connected.


In [3]:
import pandas as pd
import numpy as np
import mlflow
import dagshub
from sklearn.preprocessing import OneHotEncoder


SELECTED_FEATURES = ['OverallQual', 'GrLivArea', 'TotalBsmtSF', 'TotalBaths', 'GarageCars', 'TotalQuality', 'BsmtFinSF1', 'LotArea', 'CentralAir_N', 'YearBuilt', 'GarageYrBlt', 'LotFrontage', 'YearRemodAdd', '2ndFlrSF', 'TotalPorchArea', 'BsmtUnfSF', 'BsmtQual', 'Fireplaces', 'OverallCond', 'GarageType_Detchd', 'OpenPorchSF', 'MSZoning_C (all)', 'KitchenQual', 'WoodDeckSF', 'BsmtFinType1', 'GarageType_Attchd', 'MasVnrArea', 'BsmtExposure', 'BedroomAbvGr', 'ExterCond', 'ExterQual', 'MSZoning_RL', 'HeatingQC', 'EnclosedPorch', 'GarageFinish_Unf', 'KitchenAbvGr', 'PavedDrive_N']

HIGHLY_SKEWED_FEATURES = ['LotArea', 'MSZoning_C (all)', 'KitchenAbvGr', 'PavedDrive_N', 'CentralAir_N', 'EnclosedPorch', 'MasVnrArea', 'OpenPorchSF', 'LotFrontage', 'BsmtFinSF1', 'WoodDeckSF', 'TotalBsmtSF', 'ExterCond', 'GrLivArea', 'BsmtExposure', 'TotalPorchArea', 'GarageType_Detchd', 'BsmtUnfSF', 'ExterQual', '2ndFlrSF', 'BsmtQual', 'MSZoning_RL', 'GarageYrBlt']

DAGSHUB_USER = 'gormo22'
REPO_NAME = 'ML-HousePricePrediction'

dagshub.init(repo_owner=DAGSHUB_USER, repo_name=REPO_NAME, mlflow=True)
tracking_uri = f"https://dagshub.com/{DAGSHUB_USER}/{REPO_NAME}.mlflow"
mlflow.set_tracking_uri(tracking_uri)

train_df = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv')

X_train = train_df.drop(['Id', 'SalePrice'], axis=1).copy()
X_test = test_df.drop(['Id'], axis=1).copy()

cols_to_drop = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'MasVnrType']
X_train = X_train.drop(columns=cols_to_drop)
X_test = X_test.drop(columns=cols_to_drop)

none_cols = ['GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'FireplaceQu']
for col in none_cols:
    X_train[col] = X_train[col].fillna('None')
    X_test[col] = X_test[col].fillna('None')

zero_cols = ['GarageYrBlt', 'GarageArea', 'GarageCars', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea']
for col in zero_cols:
    X_train[col] = X_train[col].fillna(0)
    X_test[col] = X_test[col].fillna(0)

neighborhood_medians = X_train.groupby('Neighborhood')['LotFrontage'].median()
X_test['LotFrontage'] = X_test.apply(lambda row: neighborhood_medians[row['Neighborhood']] if pd.isna(row['LotFrontage']) else row['LotFrontage'], axis=1)
X_train['LotFrontage'] = X_train.apply(lambda row: neighborhood_medians[row['Neighborhood']] if pd.isna(row['LotFrontage']) else row['LotFrontage'], axis=1)

mode_cols = ['MSZoning', 'Electrical', 'KitchenQual', 'Exterior1st', 'Exterior2nd', 'SaleType', 'Functional', 'Utilities']
for col in mode_cols:
    train_mode = X_train[col].mode()[0]
    X_test[col] = X_test[col].fillna(train_mode)
    X_train[col] = X_train[col].fillna(train_mode)

for df in [X_train, X_test]:
    df['MSSubClass'] = df['MSSubClass'].astype(str)
    df['MoSold'] = df['MoSold'].astype(str)
    df['YrSold'] = df['YrSold'].astype(str)

    quality_map = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
    qual_cols = ['FireplaceQu', 'BsmtQual', 'BsmtCond', 'GarageQual', 'GarageCond', 'ExterQual', 'ExterCond', 'HeatingQC', 'KitchenQual']
    for col in qual_cols:
        df[col] = df[col].map(quality_map).fillna(0).astype(int)

    exposure_map = {'None': 0, 'No': 1, 'Mn': 2, 'Av': 3, 'Gd': 4}
    df['BsmtExposure'] = df['BsmtExposure'].map(exposure_map).fillna(0).astype(int)

    fin_map = {'None': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6}
    df['BsmtFinType1'] = df['BsmtFinType1'].map(fin_map).fillna(0).astype(int)
    df['BsmtFinType2'] = df['BsmtFinType2'].map(fin_map).fillna(0).astype(int)

def engineer_features(df):
    df = df.copy()
    df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
    df['TotalBaths'] = df['FullBath'] + (0.5 * df['HalfBath']) + df['BsmtFullBath'] + (0.5 * df['BsmtHalfBath'])
    df['TotalPorchArea'] = df['WoodDeckSF'] + df['OpenPorchSF'] + df['EnclosedPorch'] + df['3SsnPorch'] + df['ScreenPorch']
    
    df['HouseAge'] = df['YrSold'].astype(int) - df['YearBuilt']
    df['RemodelAge'] = df['YrSold'].astype(int) - df['YearRemodAdd']
    df['GarageAge'] = df['YrSold'].astype(int) - df['GarageYrBlt']
    
    df['HouseAge'] = df['HouseAge'].apply(lambda x: x if x > 0 else 0)
    df['RemodelAge'] = df['RemodelAge'].apply(lambda x: x if x > 0 else 0)
    df['TotalQuality'] = df['OverallQual'] + df['OverallCond']
    df['HasPool'] = (df['PoolArea'] > 0).astype(int)
    df['Has2ndFloor'] = (df['2ndFlrSF'] > 0).astype(int)
    df['HasGarage'] = (df['GarageArea'] > 0).astype(int)
    df['HasBsmt'] = (df['TotalBsmtSF'] > 0).astype(int)
    df['HasFireplace'] = (df['Fireplaces'] > 0).astype(int)
    df['IsRemodeled'] = (df['YearRemodAdd'] != df['YearBuilt']).astype(int)
    return df

X_train = engineer_features(X_train)
X_test = engineer_features(X_test)

cat_cols = X_train.select_dtypes(include=['object']).columns
num_cols_test = X_test.drop(columns=cat_cols)

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
ohe.fit(X_train[cat_cols])

encoded_cat_test = ohe.transform(X_test[cat_cols])
encoded_cat_test_df = pd.DataFrame(encoded_cat_test, columns=ohe.get_feature_names_out(cat_cols))

X_test_encoded = pd.concat([num_cols_test.reset_index(drop=True), encoded_cat_test_df.reset_index(drop=True)], axis=1)

X_test_final = X_test_encoded[SELECTED_FEATURES].copy()

for col in HIGHLY_SKEWED_FEATURES:
    if col in X_test_final.columns:
        X_test_final[col] = np.log1p(X_test_final[col])

print("Test data preprocessed.")

registered_model_uri = 'models:/LightGBM/2' 
print(f"Loading model from Registry: {registered_model_uri}")

loaded_model = mlflow.pyfunc.load_model(registered_model_uri)
print("Model loaded successfully.")

log_predictions = loaded_model.predict(X_test_final)
final_predictions = np.expm1(log_predictions)

submission = pd.DataFrame({
    'Id': test_df['Id'],
    'SalePrice': final_predictions
})

submission.to_csv('submission.csv', index=False)
print("submission.csv generated successfully.")

Initialized MLflow to track repo "gormo22/ML-HousePricePrediction"

Repository gormo22/ML-HousePricePrediction initialized!

Test data preprocessed.
Loading model from Registry: models:/LightGBM/2


2026/04/14 22:06:02 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - cupy-cuda12x (current: uninstalled, required: cupy-cuda12x==14.0.1)
 - distributed-ucxx-cu12 (current: uninstalled, required: distributed-ucxx-cu12==0.48.0)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.


Model loaded successfully.
submission.csv generated successfully.
